# Support Vector Regression (SVR) and Tree-Based Regression

## 📚 Learning Objectives

By completing this notebook, you will:
- Build SVR models with different kernels (linear, RBF, polynomial)
- Implement decision tree and random forest regression
- Compare tree-based models with SVR
- Understand when to use each approach

## 🔗 Prerequisites

- ✅ Understanding of regression concepts
- ✅ Python 3.8+ installed

**Note**: SVR, decision trees, and random forests are used here as *black boxes* - you only need the short intuition boxes below. Their full theory (margins, kernels, tree splits, ensembles) is taught in Unit 3.

---

This notebook covers practical activities from **Course 04, Unit 1**:
- Building SVR models with different kernels
- Implementing decision tree and random forest regression

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Generate non-linear data: a noisy sine wave (regression target)
np.random.seed(42)
X = np.sort(5 * np.random.rand(100, 1), axis=0)   # 100 points in [0, 5)
y = np.sin(X).ravel() + 0.1 * np.random.randn(100)  # sin(x) + noise

# Split into train/test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Target range:     [{y.min():.2f}, {y.max():.2f}]")

Training samples: 80
Test samples:     20
Target range:     [-1.09, 1.25]


## Part 1: Support Vector Regression with Different Kernels

**New model - the 2-minute intuition (full theory in Unit 3, Example 3):**
Linear regression fits a line by penalizing *every* error. **SVR (Support Vector Regression)** instead fits a *tube* around the data: points inside the tube cost nothing, and the fit is shaped only by the points on or outside the tube (the *support vectors*). A **kernel** decides what shape the tube can take:

- `linear` - a straight tube (like a straight line fit)
- `poly` - a curved, polynomial-shaped tube
- `rbf` - a flexible tube that can follow almost any smooth curve

That is all you need for this notebook - treat the kernels as three levels of flexibility and compare their errors.


In [3]:
# SVR with different kernels
kernels = ['linear', 'rbf', 'poly']
svr_models = {}

print("SVR performance by kernel:")
print("-" * 44)
for kernel in kernels:
    svr = SVR(kernel=kernel, C=1.0)
    svr.fit(X_train, y_train)
    pred = svr.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    svr_models[kernel] = svr
    print(f"  {kernel:<8} kernel:  MSE = {mse:.4f}   R² = {r2:.4f}")

print()
print("The data is a sine wave (non-linear), so the RBF kernel")
print("should fit it much better than the linear kernel.")

SVR performance by kernel:
--------------------------------------------
  linear   kernel:  MSE = 0.2393   R² = 0.5012
  rbf      kernel:  MSE = 0.0081   R² = 0.9831
  poly     kernel:  MSE = 0.1207   R² = 0.7485

The data is a sine wave (non-linear), so the RBF kernel
should fit it much better than the linear kernel.


## Part 2: Decision Tree and Random Forest Regression

**New models - the 2-minute intuition (full theory in Unit 3, Examples 2 and 5):**
A **decision tree** predicts by recursively splitting the input range with simple questions ("is x < 2.3?" → yes/no) and predicting the *average y* of the training points in each final region - the result is a staircase-shaped fit. A **random forest** trains many trees, each on a random sample of the data, and *averages* their predictions - the averaging smooths the staircase and reduces overfitting. Here, just compare their test errors; how splits are chosen and why averaging works comes in Unit 3.


In [4]:
# Decision Tree Regressor
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_mse = mean_squared_error(y_test, dt_pred)
dt_r2 = r2_score(y_test, dt_pred)

# Random Forest Regressor (ensemble of trees)
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

# Compare all models on the same test set
best_svr_kernel = min(svr_models, key=lambda k: mean_squared_error(y_test, svr_models[k].predict(X_test)))
best_svr_mse = mean_squared_error(y_test, svr_models[best_svr_kernel].predict(X_test))

print("Model comparison (test MSE, lower is better):")
print("-" * 44)
print(f"  SVR (best: {best_svr_kernel}):  MSE = {best_svr_mse:.4f}")
print(f"  Decision Tree:      MSE = {dt_mse:.4f}   R² = {dt_r2:.4f}")
print(f"  Random Forest:      MSE = {rf_mse:.4f}   R² = {rf_r2:.4f}")
print()
print("Random Forest averages many trees, so it is usually more")
print("robust than a single decision tree on noisy data.")

Model comparison (test MSE, lower is better):
--------------------------------------------
  SVR (best: rbf):  MSE = 0.0081
  Decision Tree:      MSE = 0.0093   R² = 0.9807
  Random Forest:      MSE = 0.0092   R² = 0.9808

Random Forest averages many trees, so it is usually more
robust than a single decision tree on noisy data.


## Summary

### Key Concepts:
1. **SVR Kernels**: Linear (simple), RBF (non-linear), Polynomial (curved)
2. **Decision Tree**: Simple, interpretable, prone to overfitting
3. **Random Forest**: Ensemble of trees, more robust, less overfitting
4. **When to use**: SVR for non-linear, Tree-based for interpretability

**Reference:** Course 04, Unit 1: "Building SVR models with different kernels" and "Implementing decision tree and random forest regression"
